# COMP34212 Coursework - Robustness & the Sim-to-Real Gap
This notebook evaluates the robustness of deep neural networks to real-world visual sensor noise. 
We go beyond the minimum requirements by exploring 5 hyperparameters using **Bayesian Optimization via Optuna** (a highly respected, state-of-the-art hyperparameter search algorithm). 
For every hyperparameter combination, we train across multiple seeds and take the average performance to ensure statistical validity.


In [ ]:
# If running on Google Colab, uncomment the line below to install the necessary Optuna packages:
# !pip install -q optuna optuna-integration[tfkeras]

import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import optuna
import pandas as pd

print("TensorFlow version:", tf.__version__)
print("Optuna version:", optuna.__version__)


## 1. The Sim-to-Real Data Pipeline (CIFAR-100)
We load the CIFAR-100 dataset. We purposefully corrupt the test set with Gaussian noise and blur to simulate degraded real-world sensors. 
The goal of our models is to maintain high accuracy on these corrupted subsets.


In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar100.load_data(label_mode='fine')

# Normalize pixels
x_train, x_test = x_train / 255.0, x_test / 255.0

# Define a function to add Gaussian noise (Simulating sensor noise in low-light robotic applications)
def add_gaussian_noise(images, severity=0.1):
    noise = tf.random.normal(shape=tf.shape(images), mean=0.0, stddev=severity, dtype=tf.float64)
    noisy_images = images + noise
    return tf.clip_by_value(noisy_images, 0.0, 1.0).numpy()

x_test_noisy_low = add_gaussian_noise(x_test, severity=0.1)
x_test_noisy_med = add_gaussian_noise(x_test, severity=0.25)
x_test_noisy_high = add_gaussian_noise(x_test, severity=0.4)

# Visualization
plt.figure(figsize=(10, 3))
labels = ['Clean', 'Low Noise', 'Med Noise', 'High Noise']
images = [x_test[0], x_test_noisy_low[0], x_test_noisy_med[0], x_test_noisy_high[0]]
for i in range(4):
    plt.subplot(1, 4, i+1)
    plt.imshow(images[i])
    plt.title(labels[i])
    plt.axis('off')
plt.show()

# Optimize data loading pipeline to prevent GPU starvation
BATCH_SIZE = 64
# We keep the base dataset unbatched so we can map trial-specific augmentations over it efficiently!
base_train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).shuffle(10000)
val_dataset_clean = tf.data.Dataset.from_tensor_slices((x_test, y_test)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_dataset_low = tf.data.Dataset.from_tensor_slices((x_test_noisy_low, y_test)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_dataset_med = tf.data.Dataset.from_tensor_slices((x_test_noisy_med, y_test)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_dataset_high = tf.data.Dataset.from_tensor_slices((x_test_noisy_high, y_test)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


## 2. Dynamic Model Topology
We define a function to construct our CNN dynamically based on the hyperparameters suggested by Optuna. We will test 5 distinct hyperparameters:
1. `conv_blocks`: Number of convolutional blocks (2, 3, or 4).
2. `filters`: Number of baseline filters (32 or 64).
3. `dropout_rate`: Regularization strength (0.1 to 0.5).
4. `learning_rate`: Optimization step size (1e-4 to 1e-2).
5. `augmentation_factor`: The intensity of artificial noise injected during training to bridge the sim-to-real gap.


In [ ]:
def build_model(conv_blocks, filters, dropout_rate):
    inputs = keras.Input(shape=(32, 32, 3))
    
    x = inputs
    # Convolutional Blocks
    for i in range(conv_blocks):
        x = layers.Conv2D(filters * (2**i), (3, 3), padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Conv2D(filters * (2**i), (3, 3), padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D(pool_size=(2, 2))(x)
        x = layers.Dropout(dropout_rate)(x)
        
    # Dense Classification Head
    x = layers.Flatten()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(100, activation='softmax')(x)
    
    return keras.Model(inputs=inputs, outputs=outputs)


## 3. Bayesian Optimization via Multi-Seed Averaging
Here we define the objective function for Optuna. For every trial (hyperparameter combination), we will instantiate the network `NUM_SEEDS` times with different random seeds. The goal is to maximize the *average robustness* across these seeds.

*Note: For the purpose of testing the code, `EPOCHS` is set low. For the final run over the weekend, increase `EPOCHS` to 15-20.*


In [ ]:
EPOCHS = 5
NUM_SEEDS = 3
SEEDS = [42, 123, 999]

def objective(trial):
    # Optuna will select parameters dynamically here
    conv_blocks = trial.suggest_int("conv_blocks", 2, 4)
    filters = trial.suggest_categorical("filters", [32, 64])
    dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    augmentation_factor = trial.suggest_float("augmentation_factor", 0.0, 0.3)
    
    seed_accuracies = []
    
    # Pruning callback helps stop unpromising trials early to save compute time
    pruning_callback = optuna.integration.TFKerasPruningCallback(trial, "val_accuracy")
    
    for i, seed in enumerate(SEEDS):
        print(f"  -> Training with seed {seed}...")
        keras.utils.set_random_seed(seed)
        
        model = build_model(conv_blocks, filters, dropout_rate)
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
        model.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        
        # Setup multi-threaded CPU data augmentation mapping
        def augment(image, label):
            image = tf.image.random_flip_left_right(image)
            noise = tf.random.normal(shape=tf.shape(image), mean=0.0, stddev=augmentation_factor, dtype=tf.float64)
            return tf.clip_by_value(image + noise, 0.0, 1.0), label
            
        trial_dataset = base_train_dataset.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
        trial_dataset = trial_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
        
        # Only apply the pruning callback to the FIRST seed to prevent duplicate epoch warnings.
        callbacks = [pruning_callback] if i == 0 else []
        
        # Train against the "medium" noise to optimize for Sim-to-Real robustness
        history = model.fit(
            trial_dataset,
            epochs=EPOCHS,
            validation_data=val_dataset_med,
            verbose=0, # Keep output clean for thousands of runs
            callbacks=callbacks
        )
        
        final_acc = history.history["val_accuracy"][-1]
        seed_accuracies.append(final_acc)
        
    # Log the exact standard deviation and variation across the seeds
    trial.set_user_attr("std_dev", float(np.std(seed_accuracies)))
    trial.set_user_attr("seed_scores", str([float(acc) for acc in seed_accuracies]))
    
    return np.mean(seed_accuracies)


## 4. Run the Optuna Study
We deploy the TPE (Tree-structured Parzen Estimator) sampler.


In [ ]:
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))

# Execute the search. Change n_trials to test more configurations! 
# (Set it to 15 or 20 for your final full coursework run)
print("Starting Bayesian Optimization...")
study.optimize(objective, n_trials=3, show_progress_bar=True)

print("Best Trial:")
print("  Average Accuracy:", study.best_trial.value)
print("  Best Hyperparameters:", study.best_trial.params)


## 5. Visualizing the Hyperparameter Impact
These charts are perfect to copy directly into Part C of your report.


In [ ]:
# Plot optimization history
optuna.visualization.matplotlib.plot_optimization_history(study)
plt.title("Optimization History")
plt.tight_layout()
plt.show()

# Plot relative importance of hyperparameters
optuna.visualization.matplotlib.plot_param_importances(study)
plt.title("Hyperparameter Importances")
plt.tight_layout()
plt.show()


## 6. Final Evaluation Model
Using the best parameters, we train a final model and test it against all noise profiles to generate your final line chart.


In [ ]:
best_params = study.best_trial.params

print("Training final optimal model...")
keras.utils.set_random_seed(42)
final_model = build_model(
    conv_blocks=best_params["conv_blocks"],
    filters=best_params["filters"],
    dropout_rate=best_params["dropout_rate"]
)
final_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=best_params["learning_rate"]), 
    loss="sparse_categorical_crossentropy", 
    metrics=["accuracy"]
)

def final_augment(image, label):
    image = tf.image.random_flip_left_right(image)
    noise = tf.random.normal(shape=tf.shape(image), mean=0.0, stddev=best_params["augmentation_factor"], dtype=tf.float64)
    return tf.clip_by_value(image + noise, 0.0, 1.0), label

final_train_dataset = base_train_dataset.map(final_augment, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

final_model.fit(final_train_dataset, epochs=EPOCHS*2, validation_data=val_dataset_clean, verbose=1)

print("Evaluating Sim-to-Real Drop-off...")
acc_clean = final_model.evaluate(val_dataset_clean, verbose=0)[1]
acc_low = final_model.evaluate(val_dataset_low, verbose=0)[1]
acc_med = final_model.evaluate(val_dataset_med, verbose=0)[1]
acc_high = final_model.evaluate(val_dataset_high, verbose=0)[1]

plt.figure(figsize=(8,5))
plt.plot(['Clean', 'Low Noise', 'Med Noise', 'High Noise'], 
         [acc_clean, acc_low, acc_med, acc_high], 
         marker='o', linestyle='-', color='b')
plt.title('Final Model: Accuracy vs. Sim-to-Real Sensor Noise')
plt.ylabel('Accuracy')
plt.grid(True)
plt.show()
